In [2]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth
from dotenv import load_dotenv
import os
import pandas as pd
import numpy as np
import requests
import warnings
from sklearn.decomposition import PCA

import plotly.express as px 
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import euclidean_distances
from scipy.spatial.distance import cdist


load_dotenv()
SPOTIPY_CLIENT_ID = os.getenv("SPOTIPY_CLIENT_ID")
SPOTIPY_CLIENT_SECRET = os.getenv("SPOTIPY_CLIENT_SECRET")
SPOTIPY_REDIRECT_URL = os.getenv("SPOTIPY_REDIRECT_URL")

In [3]:
scope = "user-library-read"
auth_manager = SpotifyOAuth(
    SPOTIPY_CLIENT_ID, 
    SPOTIPY_CLIENT_SECRET, 
    redirect_uri=SPOTIPY_REDIRECT_URL, 
    scope=scope
)

sp = spotipy.Spotify(auth_manager=auth_manager)

In [10]:
pages = 50
limit = 50
offset = 0
track_list = []

playlist = sp.playlist("37i9dQZF1DXb3m918yXHxA")

HTTP Error for GET to https://api.spotify.com/v1/playlists/37i9dQZF1DXb3m918yXHxA with Params: {'fields': None, 'market': None, 'additional_types': 'track'} returned 404 due to Resource not found


SpotifyException: http status: 404, code: -1 - https://api.spotify.com/v1/playlists/37i9dQZF1DXb3m918yXHxA?additional_types=track:
 Resource not found, reason: None

In [57]:
track_df = pd.DataFrame(track_list)
num_tracks = len(track_df)
divisions = num_tracks // 40
remainder = num_tracks % 40
print(num_tracks)
print(divisions)

944
23


## Get track data from ReccoBeats

In [58]:
div = 0
feature_list = []
headers = {
    'Accept': 'application/json'
}

for i in range(divisions):
    if i == divisions - 1:
        track_ids = track_df['id'][div: div + remainder]
    else:
        track_ids = track_df['id'][div: div + 40]
    
    id_string = ",".join(track_ids)

    url = f"https://api.reccobeats.com/v1/audio-features?ids={id_string}"

    response = requests.request("GET", url, headers=headers)
    features = response.json()["content"]

    for feature in features:
        feature_list.append(feature)
    
    
    div += 40

In [59]:
feature_df = pd.DataFrame(feature_list)
feature_df['id'] = feature_df['href'].str.removeprefix("https://open.spotify.com/track/")

In [60]:
df = pd.merge(track_df, feature_df, on='id', how='inner')

In [61]:
df['rankings'] = [-(i - len(df) + 1) / (len(df) - 1) for i in range(len(df))]
df['album'] = [album.get("name") for album in df.album]

In [62]:
song_cluster_pipeline = Pipeline(
    [
        ('scaler', StandardScaler()), 
        ('kmeans', KMeans(n_clusters=4, verbose=False))
    ], 
    verbose=False
)
X = df[['duration_ms', 'acousticness', 'danceability', 'energy', 'instrumentalness', 
        'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'rankings']]
number_cols = list(X.columns)
song_cluster_pipeline.fit(X)
song_cluster_labels = song_cluster_pipeline.predict(X)
df['cluster_label'] = song_cluster_labels

warnings.filterwarnings('ignore')

In [63]:
pca_pipeline = Pipeline([('scaler', StandardScaler()), ('PCA', PCA(n_components=2))])
song_embedding = pca_pipeline.fit_transform(X)
projection = pd.DataFrame(columns=['x', 'y'], data=song_embedding)
projection['title'] = df['name']
projection['album'] = df['album']
projection['cluster'] = df['cluster_label'].astype("int")

fig = px.scatter(
    projection, x='x', y='y', color='cluster', hover_data=['title', 'album'])
fig.show()